# 20题整体评测 · step by step

与整体版同一条算子链：准备→判据核验／冻结→作答→逐条judge→配对汇总。每个cell直接写算子，说明输入、输出与作用。**默认只读，不调用模型。**

In [ ]:
from pathlib import Path
import sys
# Works from repo root, curation/pipeline_v2, or another notebook working directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'curation/pipeline_v2').is_dir()),
            Path('/yzp/zhaozy/yangzepeng/0905/demiwtg'))
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))


In [ ]:
from pathlib import Path
from demiflow.standalone import local_data
from curation.preparation.records import run_lock, rows, digest, open_stage_dataset, from_stage_row
from curation.preparation.publication_sources import combined_input_records
from curation.evaluation.native.contracts import EvaluationFiles, verify_run, frozen_stage, BATCH, default_config
from curation.evaluation.native.operators import normalize_question, DeliverKnowledge, RetrieveKnowledge, BuildAnswerJobs, backend_partition, group_answers, verify_result
from curation.evaluation.native.adapters import GenerateImage, RunBackendGraph, validate_execution_config
from curation.evaluation.native.runtime import graph_version
from curation.evaluation.native.prompting import prompt_config, prompt_responses
from curation.evaluation.native.rubrics import PrepareRubric, ApplyRubric, freeze_rubrics, verify_rubrics, verify_rubric_inputs
from curation.evaluation.native.judging import PrepareJudge, ApplyJudge, verify_answer_assets
from curation.evaluation.native.scores import compare_group, aggregate


In [ ]:
# 显式输入和配置；这些赋值不创建run，不请求模型。
QUESTIONS = None  # 填写冻结题目阶段的 DatasetRef
KNOWLEDGE_RUNS = []  # 文章发布的 DatasetRef 或 release_id
VISUAL_RUNS = []  # 独立视觉发布；按概念与文章组合
RUN = BATCH / 'evaluation20_20260919_v2'
CONFIG = default_config()
# prepare前按已分配资源填 CONFIG['cuda']，例如每个backend一个设备编号。
# 配置随输入冻结；已prepare后再改配置须使用新的RUN目录。
# 默认None；不会沿用旧脚本占用/释放其他生产任务的GPU。
EXECUTE = False
THROUGH = 'prepare'  # prepare零调用；rubrics审核清单；generate作答；judge完整评测。
# CONFIG['judge']['mode']='offline' 默认物化/收录Astra隔离请求，缺回复时pending。
# HTTP模式改为'http'并显式配置model/base_url/api_key_env；使用相同context。
# 已有答案仅续判：run_judging(RUN)，或CLI加 --judge-only；不会加载图像模型。

PREPARE = False
REVIEW_RUBRICS = False
GENERATE = False
JUDGE = False


In [ ]:
if PREPARE:
    config = CONFIG
    files = EvaluationFiles(RUN, QUESTIONS, KNOWLEDGE_RUNS, config, graph_version(), VISUAL_RUNS)
    pack, options = prompt_config(RUN, config)
    data = local_data(prompt_packs={'evaluation.yaml': pack}, prompt_options=options, max_prompt_requests=config['judge']['max_calls'])

环境内的图像生成链也由demiflow调度；实现与整体版一致。

In [ ]:
def run_backend(run, backend, actor_factory=GenerateImage):
    # Each backend process executes native demiflow, not an external job loop.
    run = Path(run).resolve()
    manifest = verify_run(run)
    config = manifest['config']
    verify_rubrics(run)  # Enforced even for direct --backend execution.
    if actor_factory is GenerateImage:
        import os
        validate_execution_config(config, [backend])
        if backend != 'gemini' and os.environ.get('CUDA_VISIBLE_DEVICES') != str(config['cuda'][backend]):
            raise ValueError('CUDA device differs from the frozen explicit configuration')
    stage = frozen_stage(run, 'jobs')
    version = digest({'manifest': manifest, 'jobs': stage, 'backend': backend})
    folder = run / 'backends' / backend
    with run_lock(folder):
        data = local_data()
        from curation.preparation.stages import EncodeStage, stage_ref, stage_uri, PIPELINE_STAGE_ROWS
        results = (open_stage_dataset(run, 'jobs', data).map(from_stage_row)
            .filter(lambda job: job['backend'] == backend)
            .map_async(actor_factory(run, backend, config))
            .map(verify_result).map(EncodeStage('results'))
            .checkpoint_lance(stage_uri(folder, 'results'), schema=PIPELINE_STAGE_ROWS, fingerprint=version))
        return {'dataset_ref': stage_ref(folder, 'results').to_dict()}



## 1 · 读入完整开发题

输入是冻结的 题目 DatasetRef，题型与数量由该文件决定。map只整理接口、检查原图；保留每题原始审核状态。题目身份由来源run＋task_id组成，避免不同run同名题冲突。输出 questions checkpoint；题目判据留作审阅，不发给作答模型。

In [ ]:
if PREPARE:
    questions = (data.from_iter(lambda: iter({**r, 'number': i, 'publication_ref': files.questions} for i, r in enumerate(rows(files.questions), 1)))
        .map(normalize_question)
    )
    questions = files.lance_checkpoint(questions, 'questions')

In [ ]:
if PREPARE:
    from curation.preparation.inspection import table
    table(['编号', '概念', '类型', '审核'], [(r['number'], r['concept'], r['question']['task_type'], r['review_status']) for r in questions.take_all()])

## 2 · 交付已发布知识

输入是显式列出的知识run的 知识发布 DatasetRef。DeliverKnowledge复用知识链最终交付接口：正文、最终配图、引用证据。仅为解析最终引用回查来源，不引入被筛掉的图片池。published保留交付缺口；catalog是检索条目表。

In [ ]:
if PREPARE:
    published = (data.from_iter(lambda: combined_input_records(files.knowledge_files, files.visual_files))
        .map(DeliverKnowledge())
    )
    published = files.lance_checkpoint(published, 'published')
    catalog = (published.flat_map(lambda row: row['materials'])
    )
    catalog = files.lance_checkpoint(catalog, 'catalog')

In [ ]:
if PREPARE:
    from curation.preparation.inspection import show_materials
    show_materials(catalog.take(5))

## 3 · 按题面检索知识

输入 questions＋catalog。仅公开instruction作为查询，与出题共享公开概念范围、标题及IDF排序，默认最多3段文字／2张图；排除编辑原图及近重复图。不拿出题者选材冒充检索，也不把隐藏考点用于排序。输出每题实际材料、检索记录和判据依据未命中记录；后者仅用于审计。检索质量尚待看case，不宣称增强组必然拿到足够知识。

In [ ]:
if PREPARE:
    retrieved = (questions.map(RetrieveKnowledge(catalog.take_all(),
            config['split_registry'], config))
    )
    retrieved = files.lance_checkpoint(retrieved, 'retrieval')

In [ ]:
if PREPARE:
    from curation.preparation.review_notebooks import display_json
    from IPython.display import display, Markdown
    display(Markdown(display_json(retrieved.take(1))))

## 4 · 展开配置条件并冻结请求

默认每题只跑同一Qwen模型的无／有知识配对，兼容模型仅在config.backends显式选用。T2I用2512且增强只收文字；编辑用2511且增强收图文。所有编辑组原图放第一张，配图另标reference；无目标图。空知识增强组标为skipped_missing_knowledge，不用重复基线冒充增强成功。输出按题目数与配置展开的计划（其中可有跳过），实际模态、漏检、删去的图片ID均保留。配对组使用同一seed；Gemini接口未设置seed。

In [ ]:
if PREPARE:
    jobs = (retrieved.flat_map(BuildAnswerJobs(config))
    )
    jobs = files.lance_checkpoint(jobs, 'jobs')
    partitions = (jobs.reduce_by_key('backend', backend_partition)
    )
    partitions = files.lance_checkpoint(partitions, 'partitions')
    files.finish()  # Seal jobs before child Dataset graphs read them.

In [ ]:
if PREPARE:
    from curation.preparation.review_notebooks import display_json
    from IPython.display import display, Markdown
    display(Markdown(display_json(jobs.take(5))))
    # JSON展示省略重复base64；request中仍存完整图像字节。

## 5 · 准备作答前的评分清单核验

输入原题、作者判据、原题实际使用的最终知识与引用，编辑另附同一原图。PrepareRubric只整理证据编号、图像角色和标准prompt请求，不调用模型。完整来源不取自某一作答条件的检索结果。输出rubric_requests，可查看所有模型将共用的核验输入；原材料、审核拒绝标记不会混进作答请求。prepare停在这里，零模型调用。

In [ ]:
if PREPARE:
    rubric_requests = (questions.map(PrepareRubric(files.run, pack, config))
    )
    rubric_requests = files.lance_checkpoint(rubric_requests, 'rubric_requests')

In [ ]:
if PREPARE:
    from curation.evaluation.native.presentation import show_prompt_requests
    show_prompt_requests(RUN, 'rubric')

## 6 · 标准prompt审核判据、归属并冻结

map_prompt_async(rubric)独立核对原判据和依据，为每条确定dimension、子项、核心级别及允许变化；复合判据可拆分但保留原id对应。没有模型答案，不事后定标准。ApplyRubric只验证响应绑定和字段引用，然后保存不可变题目评分包。语义问题由prompt报告，不新增知识正误程序规则。ready不等于题目有效：invalid_question等审查结论随包保留；pending／材料不足保留待办，所有题的清单冻结后才允许作答。

In [ ]:
if PREPARE and REVIEW_RUBRICS:
    verify_rubric_inputs(rubric_requests.take_all())
    rubrics = (rubric_requests
        .map_prompt_async('rubric', config='evaluation.yaml',
            inputs={'instructions': 'prompt_instructions', 'payload': 'prompt_payload', 'images': 'prompt_images'},
            output='rubric_result', call_output='rubric_call', error_output='rubric_error',
            when=lambda r: r.get('rubric_status') == 'prepared', concurrency=1, queue_depth=1)
        .map(ApplyRubric(files.run, config))
    )
    rubrics = files.lance_checkpoint(rubrics, 'rubrics', extra=prompt_responses(files.run, 'rubric'))
    rubrics_ready = freeze_rubrics(files.run, rubrics.take_all())
    files.finish()

In [ ]:
if PREPARE and REVIEW_RUBRICS:
    from curation.evaluation.native.presentation import show_rubrics
    show_rubrics(RUN)

## 7 · 用demiflow执行模型分区

输入四个backend分区。map_async串行启动对应Python环境内的同一份原生Dataset作答链（下方run_backend）：read_records → filter → map_cached(GenerateImage) → map(verify_result) → checkpoint。GenerateImage仅实现一次模型调用；demiflow负责逐条调度、缓存和续跑。Python环境隔离解决BAGEL与Qwen的依赖差异。CUDA须在执行前显式填入可用设备，不关闭服务、不借卡、不恢复预标注。

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE:
    validate_execution_config(config, [r['backend'] for r in partitions.take_all()])
    completed = (partitions.map_async(RunBackendGraph(files.run, config))
    )
    completed = files.lance_checkpoint(completed, 'backend_results')

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE:
    print(completed.take_all())

## 8 · 收集作答与原始对照

输入各backend结果checkpoint；核对文件哈希和实际图片，再以题目ID聚合五组结果。保留generated／model_failure／infra_failure／unsupported_input／interrupted／skipped_missing_knowledge。原样返回原图仍是可判结果，交给judge评完成与保持。generate可停在这里，之后可judge-only续跑，不重新启动图像模型。

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE:
    result_files = completed.take_all()
    answers = data.from_iter(lambda: (verify_result(row) for record in result_files for row in rows(record['dataset_ref'])))
    answers = files.lance_checkpoint(answers, 'answers')
    comparison = (answers.reduce_by_key('task_id', group_answers)
    )
    comparison = files.lance_checkpoint(comparison, 'comparison')
    state = files.finish()

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE:
    from curation.evaluation.native.presentation import show_answers
    show_answers(RUN)

## 9 · 准备盲判请求

输入真实作答图与作答前冻结的评分包。PrepareJudge按类型读取完整新prompt，T2I附RESULT，编辑附BEFORE／AFTER，再附固定EVIDENCE配图；文件匿名，不传模型、资料条件或其他答案。所有五组共用同一rubric与证据哈希。确认没有结果的模型失败可直接记任务0，其余未知；下载失败、缺知识和不支持均不伪造分数。输出judge_requests，native offline与HTTP使用完全相同的messages。

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE and JUDGE:
    verify_answer_assets(answers.take_all())
    judge_requests = (answers.map(PrepareJudge(files.run, pack, config))
    )
    judge_requests = files.lance_checkpoint(judge_requests, 'judge_requests')

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE and JUDGE:
    from curation.evaluation.native.presentation import show_prompt_requests
    show_prompt_requests(RUN, 'judge')

## 10 · 标准prompt逐条判分

map_prompt_async(judge)执行同一份可迁移prompt。判官逐条输出图像证据并回指评分项：T2I保留10／8／4项，编辑保留类型三维；ApplyJudge检查id／维度／来源引用及可空档位，保存原始响应。T2I分别换算三线，适用项null不当N/A剔除；编辑保留三项独立分，无总分或封顶。pending、调用错误、无效响应分别保留，不当模型作答失败。

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE and JUDGE:
    scored = (judge_requests
        .map_prompt_async('judge', config='evaluation.yaml',
            inputs={'instructions': 'prompt_instructions', 'payload': 'prompt_payload', 'images': 'prompt_images'},
            output='judge_result', call_output='judge_call', error_output='judge_error',
            when=lambda r: r.get('judge_status') == 'prepared', concurrency=1, queue_depth=1)
        .map(ApplyJudge(files.run, config))
    )
    scored = files.lance_checkpoint(scored, 'scores', extra=prompt_responses(files.run, 'judge'))

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE and JUDGE:
    from curation.evaluation.native.presentation import show_answers
    show_answers(RUN, scored.reduce_by_key("task_id", group_answers).take_all())

## 11 · 配对消融与闭源参考差距

按同题BAGEL／Qwen有无知识配对，逐维计算增强−基线。Gemini只有无知识组，用同题三方共同可判集合报告开源增强前后与它的差距。编辑按类型分别汇总，Qwen文字／图文模态与机器审核通过／拒绝组分开；不输出跨维总分或知识应用分。任一条件发现题目无效，整题退出正式配对但保留全部图文；确认模型失败仍进入任务失败统计，基础设施异常／不可判列明原因和分母。输出evaluation、summary与latest，可检查逐条判据和原始响应。

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE and JUDGE:
    evaluation = (scored.reduce_by_key('task_id', group_answers).map(compare_group)
    )
    evaluation = files.lance_checkpoint(evaluation, 'evaluation')
    summary_value = aggregate(evaluation.take_all())
    summary = (data.from_iter(lambda: iter([summary_value]))
    )
    summary = files.lance_checkpoint(summary, 'summary')
    state = files.finish()

In [ ]:
if PREPARE and REVIEW_RUBRICS and rubrics_ready and GENERATE and JUDGE:
    from curation.evaluation.native.presentation import show_evaluation_summary, show_answers
    show_evaluation_summary(RUN)
    show_answers(RUN)